In [1]:

import pandas as pd
import numpy as np
import time

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import svm

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

from sklearn.pipeline import Pipeline

In [2]:

data = pd.read_csv("diabetes.csv")

print(data.head())

print("\nShape of Dataset :", data.shape)

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Shape of Dataset : (768, 9)


In [3]:
#  INPUT AND OUTPUT

X = data.drop("Outcome", axis=1)

y = data["Outcome"]


In [4]:
print("\nInput Shape :", X.shape)

print("Output Shape :", y.shape)


Input Shape : (768, 8)
Output Shape : (768,)


In [5]:
X

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33
...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63
764,2,122,70,27,0,36.8,0.340,27
765,5,121,72,23,112,26.2,0.245,30
766,1,126,60,0,0,30.1,0.349,47


In [6]:
y

0      1
1      0
2      1
3      0
4      1
      ..
763    0
764    0
765    0
766    1
767    0
Name: Outcome, Length: 768, dtype: int64

In [7]:

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42,  stratify=y)

print("\nTraining Data :", X_train.shape)

print("Testing Data  :", X_test.shape)


Training Data : (614, 8)
Testing Data  : (154, 8)


In [8]:

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [9]:

clf = svm.SVC()

start_time = time.time()

clf.fit(X_train, y_train)

end_time = time.time()

print("\nTraining Completed")

print("Execution Time :", round(end_time - start_time, 3),  "seconds")


Training Completed
Execution Time : 0.028 seconds


In [10]:

y_pred = clf.predict(X_test)

In [11]:
print("\nMODEL EVALUATION")
print("Accuracy  :", accuracy_score(y_test, y_pred)*100)
print("Precision :", precision_score(y_test, y_pred))
print("Recall    :", recall_score(y_test, y_pred))
print("F1 Score  :", f1_score(y_test, y_pred))


MODEL EVALUATION
Accuracy  : 75.32467532467533
Precision : 0.66
Recall    : 0.6111111111111112
F1 Score  : 0.6346153846153846


In [12]:

print("\nCONFUSION MATRIX")

cm = confusion_matrix(y_test, y_pred)
print(cm)


CONFUSION MATRIX
[[83 17]
 [21 33]]


In [13]:

print("\nCROSS VALIDATION")

cv_scores = cross_val_score(
    clf,
    X_train,
    y_train,
    cv=5
)

print("Cross Validation Scores :", cv_scores)

print("Average CV Score :",  np.mean(cv_scores))


CROSS VALIDATION
Cross Validation Scores : [0.73170732 0.7398374  0.77235772 0.77235772 0.75409836]
Average CV Score : 0.7540717046514728


In [14]:
# STRATIFIED K-FOLD

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
)


In [15]:
cv

StratifiedKFold(n_splits=5, random_state=None, shuffle=True)

In [16]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'gamma': ['scale', 'auto', 0.01, 0.001],
    'degree': [2, 3, 4]
}

In [17]:
param_grid

{'C': [0.01, 0.1, 1, 10],
 'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
 'gamma': ['scale', 'auto', 0.01, 0.001],
 'degree': [2, 3, 4]}

In [18]:

grid = GridSearchCV(
    estimator=svm.SVC(),
    param_grid=param_grid,
    cv=cv,
    n_jobs=-1,
)

start_time = time.time()

grid.fit(X_train, y_train)

end_time = time.time()

print("\nGRID SEARCH COMPLETED")
print("Execution Time :", round(end_time - start_time, 3),"seconds")


GRID SEARCH COMPLETED
Execution Time : 15.081 seconds


In [19]:

print("\nBEST PARAMETERS")
print(grid.best_params_)

print("\nBEST SCORE")
print(grid.best_score_)


BEST PARAMETERS
{'C': 10, 'degree': 2, 'gamma': 0.01, 'kernel': 'sigmoid'}

BEST SCORE
0.7883246701319472


In [20]:

random_search = RandomizedSearchCV(
    estimator=svm.SVC(),
    param_distributions=param_grid,
    n_iter=10,
    cv=cv,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("\nRANDOMIZED SEARCH BEST PARAMETERS")

print(random_search.best_params_)


RANDOMIZED SEARCH BEST PARAMETERS
{'kernel': 'linear', 'gamma': 0.001, 'degree': 3, 'C': 10}


In [21]:
print("\nRANDOMIZED SEARCH BEST PARAMETERS")

print(random_search.best_params_)

print("\nBEST SCORE")

print(random_search.best_score_)


RANDOMIZED SEARCH BEST PARAMETERS
{'kernel': 'linear', 'gamma': 0.001, 'degree': 3, 'C': 10}

BEST SCORE
0.781740637078502


In [22]:

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', svm.SVC())
])

pipeline_param_grid = {
    'svc__C': [0.1, 1, 10],
    'svc__kernel': ['linear', 'rbf'],
    'svc__gamma': ['scale', 'auto']
}

pipeline_grid = GridSearchCV(
    pipeline,
    pipeline_param_grid,
    cv=cv,
    n_jobs=-1
)

pipeline_grid.fit(X_train, y_train)

print("\nPIPELINE BEST PARAMETERS")
print(pipeline_grid.best_params_)

print("\nPIPELINE BEST SCORE")
print(pipeline_grid.best_score_)



PIPELINE BEST PARAMETERS
{'svc__C': 0.1, 'svc__gamma': 'scale', 'svc__kernel': 'linear'}

PIPELINE BEST SCORE
0.7736771957883514
